# Teach an AI to play DOOM

This notebook trains an AI (PPO reinforcement learning) to play DOOM with [ViZDoom](https://vizdoom.farama.org/),
using the code in this repo's `ai/` folder and the free Freedoom game data.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then run the cells top to bottom.

How it learns: it plays thousands of games at once, sees the screen, presses buttons, and gets reward for
kills, staying alive and exploring, and a penalty for dying. Early on it moves randomly; improvement shows up
as rising `ep_rew_mean` in the training log.

In [ ]:
!nvidia-smi -L

## 1. Install

In [ ]:
!git clone -q https://github.com/n1th1n-19/DOOM.git /content/DOOM 2>/dev/null || git -C /content/DOOM pull -q
%cd /content/DOOM/ai
!pip -q install -r requirements.txt

## 2. Save progress to Google Drive (recommended)

Colab sessions end after a few hours. Checkpoints saved to Drive let you **resume** later.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/doom-ai/runs'
import os; os.makedirs(OUT, exist_ok=True)

## 3. Choose what to learn

Start small and move up. Each step builds on the last.

| Step | `SCENARIO` / level | What it teaches | Steps to try |
| --- | --- | --- | --- |
| 1 | `defend_the_center` | Turn and shoot monsters coming at you | 300k (about 10 min) |
| 2 | `deadly_corridor` | Run a corridor, killing monsters on the way | 2M |
| 3 | a real level, `E1M1` of Freedoom | Explore, fight, open doors | 5M and up (hours to days) |

In [ ]:
SCENARIO = 'defend_the_center'    # defend_the_center, deadly_corridor, basic, health_gathering
LEVEL    = None                   # set to 'E1M1' to train on a real Freedoom level instead
STEPS    = 300_000
ENVS     = 4                      # parallel games; Colab has 2-4 CPU cores

NAME = LEVEL or SCENARIO
RUN  = f'{OUT}/{NAME}'
WAD  = '/content/DOOM/web/wads/freedoom1.wad'
extra = f'--wad {WAD} --map {LEVEL}' if LEVEL else f'--scenario {SCENARIO}'
print(NAME, '->', RUN)

## 4. Quick check: a random agent (should run, and score badly)

In [ ]:
from doom_env import DoomEnv
env = DoomEnv(wad=WAD, map_name=LEVEL) if LEVEL else DoomEnv(scenario=SCENARIO)
obs, _ = env.reset(); total = 0
for _ in range(300):
    obs, r, done, _, info = env.step(env.action_space.sample()); total += r
    if done: obs, _ = env.reset()
print('random agent reward over 300 steps:', round(total, 1), info)
env.close()

## 5. Train

Re-run this cell with `--resume` added to continue after a disconnect (it loads the newest checkpoint from Drive).

In [ ]:
!python train.py {extra} --steps {STEPS} --envs {ENVS} --out {RUN}

## 6. Watch the learning curve

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {RUN}

Look at `rollout/ep_rew_mean` (average reward per game). It should trend upward.

In [ ]:
!python play.py --model {RUN}/final.zip {extra} --episodes 3 --out /content/demo.mp4
from IPython.display import Video
Video('/content/demo.mp4', embed=True, width=480)

## What to do next

* **Reward not rising?** Train longer, or use a smaller scenario first. Real levels need millions of steps.
* **Change what it values** in `doom_env.py` (the reward lines in `step`): more reward for kills makes it
  aggressive, more for exploring makes it wander.
* **Harder difficulty:** add `--skill 4` (Ultra-Violence).
* **Continue later:** run the train cell again with `--resume`.

Honest limit: this learns a skill like "survive and shoot" well. Playing whole levels (finding keys, using switches
in the right order) from pixels alone is an open problem and may not work, even after days.